# Skenario 6 — LDA (Linear Discriminant Analysis) & Komparasi Final

**Notebook terakhir** dalam seri eksperimen data mining Biological Age Predictive.

### LDA vs PCA
| Aspek | PCA (B.5) | LDA (B.6) |
|-------|----------|----------|
| Tipe | **Unsupervised** | **Supervised** |
| Tujuan | Maksimalkan **varians total** | Maksimalkan **separasi antar kelas** |
| Komponen | Sebanyak fitur (35) | Maks **n_classes - 1** = 3 |
| Formula | Eigen dari matriks kovarians | Eigen dari S_w⁻¹ × S_b |

Dimana:
- **S_w** = scatter matrix *within-class* (variasi dalam kelas)
- **S_b** = scatter matrix *between-class* (variasi antar kelas)
- LDA mencari arah yang memaksimalkan rasio S_b/S_w

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Library berhasil di-import ✅')

## 1. Load Data & Hasil Sebelumnya

In [ ]:
X_train = pd.read_csv('preprocessed_data/X_train.csv')
X_test  = pd.read_csv('preprocessed_data/X_test.csv')
y_train = pd.read_csv('preprocessed_data/y_train.csv').squeeze()

prev = {}
for name, fname in [('Baseline','baseline_results.csv'), ('Filter','filter_method_results.csv'),
                     ('Wrapper','wrapper_method_results.csv'), ('Embedded','embedded_method_results.csv'),
                     ('PCA','pca_results.csv')]:
    prev[name] = pd.read_csv(fname, index_col='model')

LABEL_NAMES = {0:'Dewasa Muda (18-35)', 1:'Dewasa (36-53)', 2:'Paruh Baya (54-71)', 3:'Lansia (72-89)'}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')

---
## Part 1 — LDA
Dengan 4 kelas Age_Group, LDA menghasilkan maksimal **3 komponen** (n_classes - 1). Ini jauh lebih kompak dari PCA (26 komponen untuk 95% variance).

In [ ]:
# Fit LDA — 3 komponen (maksimal untuk 4 kelas)
lda = LinearDiscriminantAnalysis(n_components=3)
X_train_lda = lda.fit_transform(X_train, y_train)  # supervised: butuh y_train
X_test_lda  = lda.transform(X_test)

print(f'Komponen LDA: {X_train_lda.shape[1]}')
print(f'Explained variance ratio: {lda.explained_variance_ratio_}')
print(f'Total: {sum(lda.explained_variance_ratio_)*100:.2f}%')

In [ ]:
# Scatter plot: PCA 2D vs LDA 2D (side-by-side)
pca_2d = PCA(n_components=2, random_state=42)
X_pca2d = pca_2d.fit_transform(X_train)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

for ax, data, title, xlbl, ylbl in [
    (axes[0], X_pca2d, 'PCA 2D',
     f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)',
     f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)'),
    (axes[1], X_train_lda[:,:2], 'LDA 2D',
     f'LD1 ({lda.explained_variance_ratio_[0]*100:.1f}%)',
     f'LD2 ({lda.explained_variance_ratio_[1]*100:.1f}%)')
]:
    for cls in range(4):
        mask = y_train == cls
        ax.scatter(data[mask, 0], data[mask, 1], c=colors[cls],
                   label=LABEL_NAMES[cls], alpha=0.4, s=20, edgecolors='none')
    ax.set_xlabel(xlbl); ax.set_ylabel(ylbl)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Perbandingan Separasi Kelas: PCA vs LDA', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('lda_vs_pca_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretasi Visual PCA vs LDA
- **PCA**: kelas saling tumpang tindih — PCA tidak tahu tentang label kelas
- **LDA**: kelas cenderung lebih terpisah karena LDA secara eksplisit **memaksimalkan jarak antar centroid kelas** sambil meminimalkan spread dalam kelas
- Meskipun masih ada overlap (wajar karena batas usia bersifat gradual), separasi di LDA terlihat lebih jelas

### Training Model dengan 3 Komponen LDA

In [ ]:
def evaluate_model(model, X, y, name):
    y_pred = cross_val_predict(model, X, y, cv=cv)
    acc  = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, average='weighted')
    rec  = recall_score(y, y_pred, average='weighted')
    f1   = f1_score(y, y_pred, average='weighted')
    print(f'\n--- {name} ---')
    print(f'Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}')
    print(classification_report(y, y_pred, target_names=[LABEL_NAMES[i] for i in range(4)]))
    return {'model': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1_score': f1, 'y_pred': y_pred}

rf_res  = evaluate_model(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), X_train_lda, y_train, 'Random Forest')
knn_res = evaluate_model(KNeighborsClassifier(n_neighbors=7, weights='distance', n_jobs=-1), X_train_lda, y_train, 'KNN (k=7)')
svm_res = evaluate_model(SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42), X_train_lda, y_train, 'SVM (RBF)')

In [ ]:
# Confusion matrix LDA
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
labels = [LABEL_NAMES[i] for i in range(4)]
for ax, res in zip(axes, [rf_res, knn_res, svm_res]):
    cm = confusion_matrix(y_train, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels,
                yticklabels=labels, linewidths=0.5, ax=ax)
    ax.set_title(res['model'], fontsize=13, fontweight='bold')
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
    ax.tick_params(axis='x', rotation=20)
plt.suptitle('Confusion Matrix — LDA (3 komponen)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('lda_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Simpan hasil LDA
lda_results = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'y_pred'}
    for r in [rf_res, knn_res, svm_res]
]).set_index('model')
lda_results.to_csv('lda_results.csv')
print('Hasil LDA tersimpan ✅')

---
## Part 2 — Komparasi PCA vs LDA

In [ ]:
print('=' * 70)
print('PERBANDINGAN PCA vs LDA')
print('=' * 70)
print(f'{"":20s} | {"PCA (95% var)":>15s} | {"LDA":>15s}')
print(f'{"Jumlah komponen":20s} | {prev["PCA"].shape[0]:>15d}* | {3:>15d}')
print(f'{"Supervised?":20s} | {"Tidak":>15s} | {"Ya":>15s}')
print(f'\n* PCA menggunakan 26 komponen untuk 95% variance\n')

print(f'{"Model":20s} | {"PCA F1":>10s} | {"LDA F1":>10s} | {"Delta":>10s}')
print('-' * 60)
for m in prev['PCA'].index:
    p = prev['PCA'].loc[m, 'f1_score']
    l = lda_results.loc[m, 'f1_score']
    d = l - p
    arrow = '🔼' if d > 0.001 else ('🔽' if d < -0.001 else '➡️')
    print(f'{m:20s} | {p:>10.4f} | {l:>10.4f} | {d:>+10.4f} {arrow}')

### Mengapa LDA Berbeda dari PCA?
- **LDA supervised**: secara eksplisit mengoptimalkan separasi kelas umur, sehingga 3 komponen LDA bisa lebih informatif daripada 26 komponen PCA
- **PCA unsupervised**: menangkap varians total yang mungkin didominasi oleh variasi yang tidak relevan untuk klasifikasi umur
- Namun, LDA membuat asumsi **linear** — jika batas kelas sangat non-linear, LDA bisa kurang optimal

---
## Part 3 — Kesimpulan Eksperimen Keseluruhan

### Tabel Rekap FINAL — 6 Skenario × 3 Model

In [ ]:
prev['LDA'] = lda_results
scenario_names = ['Baseline', 'Filter', 'Wrapper', 'Embedded', 'PCA', 'LDA']

print('=' * 115)
print('REKAP FINAL F1-SCORE — SELURUH EKSPERIMEN')
print('=' * 115)

header = f'{"Model":20s}'
for s in scenario_names:
    header += f' | {s:>12s}'
header += ' |     Best'
print(header)
print('-' * 115)

for m in prev['Baseline'].index:
    vals = [prev[s].loc[m, 'f1_score'] for s in scenario_names]
    best_val = max(vals)
    best_idx = vals.index(best_val)
    line = f'{m:20s}'
    for i, v in enumerate(vals):
        flag = ' 🏆' if i == best_idx else '   '
        line += f' | {v:>8.4f}{flag}'
    line += f' | {scenario_names[best_idx]}'
    print(line)

print('\n' + '=' * 115)

In [ ]:
# Visualisasi final 6 skenario
fig, ax = plt.subplots(figsize=(15, 7))
models = prev['Baseline'].index.tolist()
x = np.arange(len(models))
w = 0.13
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974', '#64B5CD']

for i, s in enumerate(scenario_names):
    vals = [prev[s].loc[m, 'f1_score'] for m in models]
    bars = ax.bar(x + i*w, vals, w, label=s, color=colors[i], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{bar.get_height():.3f}', ha='center', fontsize=6, fontweight='bold', rotation=90)

ax.set_xticks(x + 2.5*w)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel('F1-Score (weighted)', fontsize=12)
ax.set_title('Rekap Final: F1-Score Seluruh Skenario', fontsize=15, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Jawaban Pertanyaan Tugas

### a. Bagaimana kondisi awal dataset?
- Dataset Biological Age Predictive dari Kaggle: 3.000 sampel training, 3.000 sampel test, 27 kolom
- Target: `Age (years)` — integer, rentang 18–89 tahun, distribusi mendekati uniform
- Terdapat missing values pada 5 kolom kategorikal (20-48%), nilai negatif pada Bone Density, kolom Blood Pressure berbentuk string

### b. Bagaimana proses data preparation?
1. Hapus kolom ID (tidak relevan)
2. Parsing `Blood Pressure (s/d)` → `BP_Systolic` + `BP_Diastolic`
3. Koreksi nilai negatif Bone Density dengan `abs()`
4. Imputasi missing values kategorikal → `'Unknown'`
5. Label Encoding (ordinal/binary) + One-Hot Encoding (nominal)
6. StandardScaler pada fitur numerik → 35 fitur final
7. **Binning** `Age (years)` → 4 kelas `Age_Group` (Dewasa Muda, Dewasa, Paruh Baya, Lansia) — seimbang (~25% per kelas)

### c. Fitur yang dipilih oleh Wrapper dan Embedded, mengapa?

**Wrapper (RFECV-RF, 25 fitur):** RFE menghapus fitur secara iteratif berdasarkan importance — fitur OHE dengan variasi rendah tereliminasi lebih dulu. Hasilnya masih cukup banyak (25 dari 35) karena RF bisa memanfaatkan banyak fitur melalui random subspace.

**Embedded (RF+XGBoost, 8 fitur):** Kedua model tree-based sepakat bahwa fitur paling penting adalah **biomarker biologis penuaan**: Bone Density, Vision Sharpness, Hearing Ability, Cognitive Function, Blood Glucose, Cholesterol, BMI, dan BP. Ini sangat masuk akal secara domain kesehatan karena indikator-indikator ini memang terdokumentasi berubah signifikan seiring usia.

### d. Bagaimana PCA dan LDA mengubah representasi data?
- **PCA**: mentransformasi 35 fitur menjadi komponen orthogonal yang memaksimalkan **varians total**. 26 PC dibutuhkan untuk menangkap 95% varians. Komponen tidak memiliki makna biologis.
- **LDA**: mentransformasi 35 fitur menjadi 3 komponen yang memaksimalkan **separasi kelas umur**. Jauh lebih kompak dan task-specific.

### e. Apakah performa meningkat atau menurun?
- **Seleksi fitur (Filter, Wrapper, Embedded)**: umumnya **meningkatkan** atau mempertahankan performa, terutama untuk KNN (+16% di Embedded)
- **PCA**: **menurunkan** semua model karena unsupervised — varians terbesar tidak selalu diskriminatif
- **LDA**: supervised, hasilnya tergantung apakah separasi linear cukup untuk dataset ini

### f. Metode mana yang paling efektif?
- **Untuk RF**: Filter Method (ANOVA + MI, 18 fitur) — peningkatan tipis dengan model yang lebih sederhana
- **Untuk KNN**: Embedded Method (8 fitur) — pengurangan dimensi drastis mengatasi curse of dimensionality
- **Untuk SVM**: Embedded Method — fitur inti biomarker cukup untuk margin optimal
- **Secara keseluruhan**: Embedded Method paling konsisten baik di semua model

### Analisis: Mengapa Setiap Model Merespons Berbeda?

| Model | Karakteristik | Respons terhadap seleksi fitur | Penjelasan |
|-------|-------------|-------------------------------|------------|
| **RF** | Bagging + random subspace | Stabil (~80-81%) | Internal feature selection via tree splits + random subsampling sudah otomatis mengabaikan fitur noise |
| **KNN** | Distance-based | Sangat sensitif (61% → 77%) | Curse of dimensionality: fitur noise mendistorsi jarak antar sampel. Mengurangi fitur = mengurangi distorsi |
| **SVM** | Margin-based | Moderat (80% → 81%) | Kernel RBF mampu menangani noise, tapi fitur yang lebih bersih menghasilkan margin yang lebih optimal |

### Mengapa PCA/LDA Berbeda dari Seleksi Fitur?
- **Seleksi fitur** mempertahankan fitur asli → model masih bekerja di ruang fitur yang familiar
- **PCA/LDA** mentransformasi ke ruang baru → tree-based models (RF) kehilangan kemampuan split yang bermakna karena komponen bersifat abstrak
- KNN dan SVM lebih cocok dengan transformasi karena mereka berbasis jarak/margin, bukan splits

---

## Rekomendasi Praktis untuk Deployment

### Pipeline yang Direkomendasikan:
**Random Forest + Filter Method (18 fitur)**

Alasan:
1. **Performa terbaik** untuk RF (F1 = 0.8126) — model terkuat secara konsisten
2. **Interpretable**: 18 fitur asli yang bermakna secara domain kesehatan
3. **Robust**: RF tidak sensitif terhadap outlier, missing values, atau variasi data baru
4. **Efisien**: 18 fitur vs 35 → pengumpulan data lebih mudah, inference lebih cepat
5. **Reproducible**: Filter method (ANOVA + MI) deterministik dan tidak tergantung model

### Alternatif jika Kecepatan Penting:
**SVM (RBF) + Embedded Method (8 fitur)** — hanya butuh 8 biomarker inti, F1 = 0.8099

### Catatan Penting:
- Akurasi ~81% untuk klasifikasi 4 kelas umur adalah hasil yang **baik** mengingat batas antar kelas umur bersifat gradual (bukan diskrit)
- Misklasifikasi sebagian besar terjadi antara kelas **bersebelahan** (misal: Dewasa↔Paruh Baya), bukan kelas yang jauh (misal: Dewasa Muda↔Lansia)
- Ini menunjukkan model sudah menangkap pola penuaan biologis dengan baik